# 03 — Pretrained audio features

Before training our own networks: how much does a large pretrained audio model already know about murmurs?

**Where this comes from.** In notebook 02 the pretrained tagger's *labels* did poorly on our audio: it is deaf
to crying, breathing and handling, because our recordings stop at 2 kHz and are heard through the body.
But a network's internal features can carry more than its labels. If they did, we could build on them instead of training
from scratch, which with ~750 patients is a real concern.

We take PANNs Cnn14 (trained on AudioSet, 2 million clips), **freeze** it, and use its internal features
(2048 per frame, mean- and max-pooled over each recording) as input to a **linear** classifier. No network is
trained here. The question is what the frozen features carry. The score is the competition's weighted
accuracy (WA), out-of-fold over the 5 folds of the 753 training patients.

`RECOMPUTE = True` re-extracts the features (GPU, about 5 min per setting) and refits the probes; `False` reads
`results/pretrained-probe/`.

In [1]:
import librosa
import numpy as np
import pandas as pd

from murmur.core.paths import results_dir
from murmur.data import circor, split
from murmur.evaluate import probe
from murmur.preprocess import audioset, spectral

RECOMPUTE = False
patients, recordings, s = circor.patients(), circor.recordings(), split.load()
train = s[s.split == "train"]
train_recs = recordings[recordings["patient_id"].isin(train.index)]

## The problem: our audio sits in a corner of the model's input

The model turns sound into 64 mel bands between 50 Hz and 14 kHz. Our recordings stop at 2 kHz, and the heart
lives below about 800 Hz. **Playing the recording faster** (declaring 4 kHz audio to be 16 or 32 kHz) moves the
heart band up into more of the model's bands, at the cost of time detail, and of the model's class names,
which stop meaning anything:

In [2]:
centres = librosa.mel_frequencies(66, fmin=50, fmax=14000)[1:-1]
rows = {}
for speed in (1, 4, 8):
    f = centres / speed      # the original frequency each band hears at this speed
    rows[f"×{speed}"] = {"bands on 25–400 Hz": int(((f >= 25) & (f < 400)).sum()),
                         "bands on 25–800 Hz": int(((f >= 25) & (f < 800)).sum()),
                         "empty bands (above 2 kHz)": int((f >= 2000).sum())}
pd.DataFrame(rows).T

,bands on 25–400 Hz,bands on 25–800 Hz,empty bands (above 2 kHz)
×1,6,13,34
×4,26,38,10
×8,36,48,0


## Results

In [3]:
path = results_dir() / "pretrained-probe" / "oof-patients.tsv"
settings = {"psd": "reference: the recording's own spectrum 25–800 Hz (no pretrained model)",
            "x1": "Cnn14, normal speed", "x4": "Cnn14, 4× speed", "x8": "Cnn14, 8× speed",
            "x4_random": "control: Cnn14 with RANDOM weights, 4× speed"}
if RECOMPUTE:
    rec_patient = train_recs["patient_id"].to_numpy()
    out = []
    for name in settings:
        if name == "psd":
            X = np.stack([10 * np.log10(p[(f >= 25) & (f <= 800)] + 1e-12)
                          for f, p in (spectral.psd(circor.audio(r)) for _, r in train_recs.iterrows())])
        else:
            speed = int(name[1])
            X = audioset.embed_all(train_recs, circor.audio, speed, random_weights=name.endswith("random"))
        out.append(probe.oof_probs(X, rec_patient, train.index, train["fold"], patients["murmur"]).add_prefix(name + ":"))
    pd.concat(out, axis=1).to_csv(path, sep="\t", index_label="patient_id")
stored = pd.read_csv(path, sep="\t", index_col="patient_id")
rows, matrices = [], {}
for name, what in settings.items():
    probs = stored.filter(like=name + ":").rename(columns=lambda c: c.split(":")[1])
    row, matrices[name] = probe.summary(name, probs, patients["murmur"])
    rows.append({**row, "what": what})
pd.DataFrame(rows).set_index("features").round(3)

,WA,AUC Present,AUC Unknown,what
features,,,,
psd,0.551,0.660,0.746,reference: the recording's own spectrum 25–800...
x1,0.609,0.741,0.879,"Cnn14, normal speed"
x4,0.655,0.795,0.840,"Cnn14, 4× speed"
x8,0.636,0.775,0.856,"Cnn14, 8× speed"
x4_random,0.608,0.772,0.778,"control: Cnn14 with RANDOM weights, 4× speed"


In [4]:
print("4× speed, patient-level confusion (rows true, columns predicted: Present, Unknown, Absent)")
matrices["x4"]

4× speed, patient-level confusion (rows true, columns predicted: Present, Unknown, Absent)


array([[100,   7,  36],
       [ 20,  13,  21],
       [137,  19, 400]])

For reference, always predicting Present scores WA 0.499.

- **The frozen features carry real information:** 0.655 at 4× speed, against 0.551 for the recording's own
  spectrum.
- **The speed-up helps Present** (AUC 0.74 → 0.80), as the band count predicted.
- **The random-weights control is the important row:** the same network with **untrained** weights already
  reaches 0.608 and most of the Present gain (AUC 0.77). Most of the benefit comes from passing the audio
  through a deep convolutional network at all; AudioSet pretraining adds about +0.05, mostly for Unknown.

## What this means

| finding | consequence |
|---|---|
| A frozen pretrained model + linear classifier: 0.655 | The bar our own CNN must beat (notebook 04: 0.768) |
| Untrained convolutions get most of the way | A CNN trained from scratch on our data is the right main model |
| Pretraining helps mostly Unknown | The pretrained model is a possible extra input, not the backbone |